In [1]:
import os
import mne
import numpy as np
import torch
from torchvision import models
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam

from src import utils

%reload_ext autoreload
%autoreload 2
%matplotlib inline


In [2]:
torch.cuda.empty_cache()
torch.cuda.is_available()

if torch.cuda.is_available():
    print('Number of GPUs: ',torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print('GPU', i, torch.cuda.get_device_name(i))

torch.cuda.set_device(1)
##nvidia-smi

Number of GPUs:  4
GPU 0 NVIDIA RTX A6000
GPU 1 NVIDIA RTX A6000
GPU 2 NVIDIA RTX A6000
GPU 3 NVIDIA RTX A6000


In [3]:
from src.dataset import EEGDataset

data_folder= '/space/gzanardini/tuh'
h_data_folder = data_folder + '/Non-epileptic EDF/'
h_data_lookup_table = data_folder + '/info_h/TUH_Epilepsy_h_lookup_table.txt'
h_labels = data_folder + '/info_h/TUH_Epilepsy_h_text_label.csv'

pat_data_folder = data_folder + '/Epilepsy EDF/'
pat_data_lookup_table = data_folder + '/info_pat/TUH_Epilepsy_pat_lookup_table.txt'
pat_labels = data_folder + '/info_pat/TUH_Epilepsy_pat_text_label.csv'

# data_folder = 'data/chb-mit'

window_length = 10 # seconds
overlap_factor = 1 # no overlap for the moment

h_dataset = EEGDataset(h_data_folder, h_data_lookup_table, h_labels, window_length, overlap_factor)
pat_dataset = EEGDataset(pat_data_folder, pat_data_lookup_table, pat_labels, window_length, overlap_factor)

do_preprocessing = True
if do_preprocessing:
    h_dataset.preprocessing()
    pat_dataset.preprocessing()

Preprocessing and segmenting EEG data:   0%|          | 0/288 [00:00<?, ?it/s]

125 1
Notch filter: low=0.472, high=0.488
Notch filter: low=0.472, high=0.488


Preprocessing and segmenting EEG data:   5%|▍         | 13/288 [02:27<52:03, 11.36s/it]

125 1
Notch filter: low=0.472, high=0.488
Notch filter: low=0.472, high=0.488


Preprocessing and segmenting EEG data:   5%|▍         | 14/288 [05:02<1:56:52, 25.59s/it]

...

125 1
Notch filter: low=0.472, high=0.488
Notch filter: low=0.472, high=0.488


Preprocessing and segmenting EEG data:   9%|▉         | 26/288 [45:30<16:53:49, 232.18s/it]

125 1
Notch filter: low=0.472, high=0.488
Notch filter: low=0.472, high=0.488


Preprocessing and segmenting EEG data:   9%|▉         | 26/288 [49:00<8:13:53, 113.10s/it] 


KeyboardInterrupt: 

'EEG labels final text labels.xlsx'   TUH_Epilepsy_h_text_label.csv
'Epilepsy EDF'			      TUH_Epilepsy_h_text_label.txt
 info_h				      TUH_Epilepsy_pat_lookup_table.txt
'Non-epileptic EDF'		      TUH_Epilepsy_pat_text_label.csv
 Readme.txt			      TUH_Epilepsy_pat_text_label.txt
 TUH_Epilepsy_h_lookup_table.txt


ResNet50 stuff

In [ ]:
resnet50 = models.resnet50(pretrained=True)
resnet50.fc = nn.Linear(in_features=2048, out_features=2, bias=True)
renset50.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)